# Predicting Goodreads Shelves from Book Descriptions

CSE 258, Assignment 2. This notebook accompanies our presentation and follows the five parts of the assignment in order: the predictive task, the data, the model, the evaluation, and related work.

On Goodreads, readers organize books into "shelves" that they name themselves. Shelves such as `historical-romance`, `paranormal`, or `m-m` work as community-assigned genre tags, and a recommender or a search page can use them to find books of a given kind. These tags only accumulate after people have read a book. A newly published book has no shelves, so a system that relies on shelves has no way to place it. This is an instance of the item cold-start problem discussed in the course.

A new book does come with a description, usually written by the publisher. Our research question is whether a model that sees only a book's description can predict which shelves readers will put it on. We study this question with a TF-IDF representation of the description and one logistic regression per shelf, a standard linear text classifier.

The short answer is that it depends on the shelf. For shelves whose books share a recognizable vocabulary, the description is a strong signal: on books the model has never seen, it reaches an F1 score of 0.81 for `manga`, 0.69 for `historical-romance`, and 0.63 for `m-m`. For shelves that readers apply inconsistently, the signal is weak, with an F1 of 0.24 for `contemporary` and 0.28 for `contemporary-romance`. Averaged over all 15 shelves, the model's macro-F1 is 0.45, compared with 0.11 for a rule that looks for the shelf's name in the text. Section 4 shows that the main reason some shelves fail is disagreement among readers about the label, and that the amount of training data plays a much smaller role.

## 1. Predictive task

### Problem statement

Each example is one book. The input is the text of its description, and the output is a vector of 15 binary labels, one for each of the 15 most common genre shelves. A book may carry several shelves or none, so this is a multi-label classification problem.

The labels come from reader behavior. For every book, the dataset lists its most popular shelves and how many times readers placed the book on each. We treat a shelf as a true label for a book when it receives at least 10% of the book's genre-related shelf votes; section 2.2 explains this choice. The definition rests on an assumption: a shelf chosen by a tenth of a book's readers reflects a shared opinion about the book, while a shelf chosen by one or two readers does not.

### Simulating cold start

The data is a single snapshot, so we cannot observe books at the moment of release. We approximate the cold-start setting by taking books whose shelves are known, hiding those shelves, and asking the model to recover them from the description alone. This approximation assumes that the descriptions of established books resemble those of new books. It also limits us to books with at least 10 genre votes, which are more established than a typical new release.

### Evaluation protocol

We split the books into training (64%), validation (16%), and test (20%) sets. Different editions of the same work usually share a description, so we split by work, which places all editions of a book in the same set. Without this grouping, the model could be tested on descriptions it had already seen during training. The validation set is used only to choose a decision threshold for each shelf, and the test set is used once, after every choice has been fixed.

We measure performance mainly with F1, the harmonic mean of precision (the share of predicted shelves that are correct) and recall (the share of true shelves that the model finds). We report micro-F1, which pools all book-shelf decisions and is therefore dominated by frequent shelves, and macro-F1, which averages the F1 of each shelf and gives every shelf equal weight. We treat macro-F1 as the primary metric because our question concerns individual shelves, including rare ones. We also report ROC AUC and average precision (AP), which measure how well the model ranks books before any threshold is applied. Accuracy is not informative for this task. Since 97.1% of book-shelf pairs are negative, a model that never predicts any shelf would be 97.1% accurate and entirely useless.

### Baselines

We compare against two baselines that involve no learning. The label-prior baseline predicts a shelf whenever more than half of the training books carry it. No shelf is that common, so it never predicts anything and scores an F1 of zero; it marks the floor that any useful model must clear. The keyword baseline predicts a shelf whenever the shelf's name, such as "historical romance", appears in the description, which tests whether the task can be solved by looking for the obvious word. We also report our own model with the default threshold of probability 0.5, to separate the contribution of the learned weights from the contribution of threshold tuning.

### Validity

We check whether the predictions can be trusted in three ways. First, the split guarantees that test books, and all other editions of them, play no role in training or in threshold selection. Second, we inspect the learned weights to confirm that the model relies on genre-related words. Third, since shelf votes are a noisy proxy for genre, we measure the noise in each shelf's labels (section 2.3) and test whether it explains where the model fails (section 4).

## 2. Data, exploration, and preprocessing

### 2.1 Source

We use the romance subset of the UCSD Goodreads dataset [1, 2], which was collected in late 2017 by crawling public Goodreads shelves. Each line of the file describes one book: its title, authors, description, publication details, rating summary, similar books, and most popular shelves with their vote counts. The file contains no user identifiers and no reviews, so the interaction-based models from the course, such as latent factor models and BPR, cannot be applied, and a content-based approach is the natural choice. The dataset is distributed for academic use only, so it is not included in this repository; the notebook reads a local copy whose path is set in `secret.env`.

The following cells load the full file and examine each field.

In [1]:
import gzip
import json
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from scipy.sparse import hstack
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit

In [2]:
load_dotenv(find_dotenv('secret.env', usecwd=True))
PATH = os.environ['GOODREADS_PATH']

In [3]:
# set LIMIT (e.g. 50_000) for a quick pass
LIMIT = None
rows = []
with gzip.open(PATH, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if LIMIT and i >= LIMIT:
            break
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
del rows
print(df.shape)

(335449, 29)


In [4]:
df.iloc[0].to_dict()

{'isbn': '',
 'text_reviews_count': '4',
 'series': [],
 'country_code': 'US',
 'language_code': '',
 'popular_shelves': [{'count': '4', 'name': 'to-read'},
  {'count': '1', 'name': 'ibooks'},
  {'count': '1', 'name': 'favorite-authors'},
  {'count': '1', 'name': 'may-2017-dr-reads'},
  {'count': '1', 'name': 'sports-romance'},
  {'count': '1', 'name': 'series-i-like'},
  {'count': '1', 'name': 'romance'},
  {'count': '1', 'name': 'favorites'},
  {'count': '1', 'name': 'f-f'},
  {'count': '1', 'name': 'hockey-romance'},
  {'count': '1', 'name': 'ebook'}],
 'asin': '',
 'is_ebook': 'true',
 'average_rating': '3.86',
 'kindle_asin': '',
 'similar_books': [],
 'description': "Secrets. Sometimes keeping them in confidence is a good thing. Other times secrets can slowly allow a woman's soul to rot.\nWhitney Beaupre has been hiding a big secret for years, one that's beginning to wear her down both on and off the ice. Pretending to be something she's not is exhausting. Wanting to be free but 

In [5]:
empty = df.map(lambda v: v in ('', None) or (isinstance(v, list) and not v))
(1 - empty.mean()).sort_values().round(3)

edition_information     0.101
asin                    0.366
isbn                    0.394
isbn13                  0.504
kindle_asin             0.534
num_pages               0.593
publication_day         0.605
similar_books           0.619
series                  0.623
publisher               0.631
publication_month       0.659
format                  0.664
language_code           0.689
publication_year        0.697
description             0.909
popular_shelves         1.000
authors                 1.000
link                    1.000
average_rating          1.000
is_ebook                1.000
country_code            1.000
text_reviews_count      1.000
url                     1.000
image_url               1.000
book_id                 1.000
ratings_count           1.000
work_id                 1.000
title                   1.000
title_without_series    1.000
dtype: float64

In [6]:
for c in ['average_rating', 'ratings_count', 'text_reviews_count', 'num_pages',
          'publication_year', 'publication_month', 'publication_day']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df[['average_rating', 'ratings_count', 'text_reviews_count', 'num_pages', 'publication_year']].describe()

,average_rating,ratings_count,text_reviews_count,num_pages,publication_year
count,335449.000000,3.354490e+05,335449.000000,198819.000000,233649.000000
mean,3.880701,4.480081e+02,37.945822,267.380552,2012.429302
std,0.376427,7.285129e+03,277.039372,311.701089,249.421360
min,0.000000,0.000000e+00,0.000000,0.000000,1.000000
25%,3.670000,1.600000e+01,3.000000,184.000000,2010.000000
50%,3.910000,5.200000e+01,9.000000,264.000000,2013.000000
75%,4.140000,1.750000e+02,25.000000,352.000000,2015.000000
max,5.000000,2.078406e+06,67203.000000,82000.000000,65535.000000


In [7]:
df['average_rating'].round(1).value_counts().sort_index()

average_rating
0.0       46
1.0       19
1.2        1
1.3        4
1.4        2
1.5        4
1.6        8
1.7       15
1.8       29
1.9       12
2.0       84
2.1       47
2.2      126
2.3      145
2.4      233
2.5      338
2.6      567
2.7      716
2.8     1349
2.9     1644
3.0     3093
3.1     3662
3.2     6741
3.3     7939
3.4    13722
3.5    14976
3.6    25643
3.7    26742
3.8    40055
3.9    34015
4.0    40837
4.1    32151
4.2    32107
4.3    19324
4.4    14607
4.5     6707
4.6     4582
4.7     1725
4.8      846
4.9      257
5.0      329
Name: count, dtype: int64

In [8]:
zero_share = (df['ratings_count'] == 0).mean()
print(f'{zero_share} share with 0 ratings')
df['ratings_count'].quantile([.1, .25, .5, .75, .9, .99])

0.004468637557423036 share with 0 ratings


0.10       5.00
0.25      16.00
0.50      52.00
0.75     175.00
0.90     591.00
0.99    5763.52
Name: ratings_count, dtype: float64

In [9]:
df[['average_rating', 'ratings_count', 'text_reviews_count', 'num_pages', 'publication_year']].corr(method='spearman')

,average_rating,ratings_count,text_reviews_count,num_pages,publication_year
average_rating,1.000000,0.088114,0.147278,0.241289,0.303357
ratings_count,0.088114,1.000000,0.848879,0.046207,-0.101965
text_reviews_count,0.147278,0.848879,1.000000,0.038564,0.142427
num_pages,0.241289,0.046207,0.038564,1.000000,-0.043116
publication_year,0.303357,-0.101965,0.142427,-0.043116,1.000000


In [10]:
df['publication_year'].value_counts().sort_index().tail(40)

publication_year
2015.0     25616
2016.0     21587
2017.0     13612
2018.0       182
2019.0        11
2020.0         1
2035.0         1
2099.0         2
2100.0         1
2102.0         2
2103.0         3
2104.0         6
2105.0         9
2106.0         4
2107.0         2
2112.0         2
2140.0         1
2616.0         1
2914.0         1
2916.0         2
2997.0         1
3016.0         1
3017.0         1
8017.0         1
20015.0        1
20017.0        1
20032.0        1
20102.0        1
20112.0        3
20113.0        1
20115.0        2
20131.0        1
20140.0        1
20141.0        1
21012.0        1
21014.0        1
21016.0        1
22012.0        1
22013.0        1
65535.0        2
Name: count, dtype: int64

In [11]:
for c in ['language_code', 'format', 'is_ebook', 'country_code']:
    print(df[c].replace('', '(empty)').value_counts().head(10), '\n')

language_code
eng        160125
(empty)    104396
en-US       18932
en-GB       14793
ita          5292
spa          5060
ind          4178
en-CA        3420
ger          3158
fre          3071
Name: count, dtype: int64 

format
(empty)                  112865
Paperback                 86188
ebook                     63500
Kindle Edition            39504
Mass Market Paperback     11924
Hardcover                  9648
Audio CD                   3573
Audible Audio              3071
Audiobook                  2213
Audio                       686
Name: count, dtype: int64 

is_ebook
true     194447
false    141002
Name: count, dtype: int64 

country_code
US    335449
Name: count, dtype: int64 



In [12]:
for c in ['authors', 'series', 'similar_books', 'popular_shelves']:
    print(c, df[c].map(len).describe()[['mean', '50%', 'max']].round(1).to_dict())

authors {'mean': 1.2, '50%': 1.0, 'max': 51.0}
series {'mean': 0.7, '50%': 1.0, 'max': 4.0}
similar_books {'mean': 9.4, '50%': 13.0, 'max': 18.0}
popular_shelves {'mean': 78.2, '50%': 100.0, 'max': 100.0}


In [13]:
tot, nb = Counter(), Counter()
for shelves in df['popular_shelves']:
    for s in shelves:
        tot[s['name']] += int(s['count'])
        nb[s['name']] += 1
pd.DataFrame({'total': tot, 'books': nb}).sort_values('total', ascending=False).head(40)

,total,books
to-read,997209055,333089
currently-reading,89012773,289536
classics,43948395,6761
favorites,39882117,189483
romance,32520924,312899
fiction,14947713,178612
books-i-own,9020178,137973
contemporary,7041696,192850
owned,6981115,187043
classic,5957605,4726


In [14]:
a = Counter(x['author_id'] for au in df['authors'] for x in au)
a_stats = pd.Series(a).describe()[['mean', '50%', 'max']].round(1).to_dict()
print(f'{len(a)} authors; books per author: {a_stats}')
s = Counter(x for se in df['series'] for x in se)
s_stats = pd.Series(s).describe()[['mean', '50%', 'max']].round(1).to_dict()
print(f'{len(s)} series; books per series: {s_stats}')

52687 authors; books per author: {'mean': 7.6, '50%': 2.0, 'max': 2417.0}
116535 series; books per series: {'mean': 1.9, '50%': 1.0, 'max': 114.0}


In [15]:
d = df['description'].replace('', pd.NA).dropna()
d_words = d.str.split().str.len().describe()[['mean', '50%', 'max']].round(0).to_dict()
print(f'{len(d)} with description; words: {d_words}')
d.iloc[:3].tolist()

304837 with description; words: {'mean': 156.0, '50%': 147.0, 'max': 3943.0}


["Secrets. Sometimes keeping them in confidence is a good thing. Other times secrets can slowly allow a woman's soul to rot.\nWhitney Beaupre has been hiding a big secret for years, one that's beginning to wear her down both on and off the ice. Pretending to be something she's not is exhausting. Wanting to be free but afraid to break out of her prison is terrifying. Seeking love but then hiding from it is crushing to the spirit, yet Whitney feels compelled to keep living the lie. Until the night Hannah Kym appears in her life. Whitney's attraction to Hannah is deep, fierce, and instantaneous. The Temple art major is everything Whitney has dreamed of and more. But those old fears keep clawing at the Venom center, keeping her locked in the closet despite the passion and affection she feels for Hannah.\nCan love finally break the shackles holding Whitney's heart and soul captive?",
 "The funny and heartwarming story of a young lady whose zeal, snobbishness, and self-satisfaction lead to s

In [16]:
ids = set(df['book_id'])
sim = [x for lst in df['similar_books'] for x in lst]
inside = sum(x in ids for x in sim) / max(len(sim), 1)
print(f'{len(sim)} links; {inside:.3f} point inside the file')

3146805 links; 0.838 point inside the file


### What the exploration shows

The file contains 335,449 books. Of these, 90.9% have a non-empty description, with a median length of 147 words, so for most books the model has a full paragraph of text to work with.

The remaining metadata is noisy. Publication year is missing for 30% of books and includes impossible values such as 1, 20015, and 65535, which are most likely data-entry errors. Rating counts are highly skewed: the median book has 52 ratings, the mean is 448, and the most-rated book has about 2.08 million. Language codes are missing for 31% of books, and Italian, Spanish, and Indonesian editions each number several thousand, so not every description is in English. Our model reads only the description, so these fields need no cleaning, but the mix of languages is a source of noise in the text itself.

The shelf table shows why the labels need preprocessing. The most-used shelves are `to-read`, `currently-reading`, and `favorites`, which record what a reader did with a book and say nothing about its genre. Genre shelves such as `historical-romance` and `paranormal` appear further down the list. Finally, 83.8% of `similar_books` links point to other books in this file, which would make a link-prediction task possible; we leave that direction for future work.

### 2.2 Constructing labels

We turn the raw shelf votes into labels in four steps. First, we remove shelves that do not describe content: reading status (`to-read`), format (`kindle`, `ebook`), ownership (`owned`), and the tags `romance` and `fiction`, which nearly every book in this file carries and which cannot distinguish one book from another. These shelves are listed in `EXCLUDE`, and we call the remaining votes content votes. Second, we keep only books with at least 10 content votes, because with fewer votes a label could be decided by two or three readers. Third, a shelf is a positive label for a book when it receives at least 10% of the book's content votes. Fourth, we select the 15 shelves that are most often positive among the training books. Selecting them from training data alone keeps the test set unseen.

The 10% threshold trades the quality of the labels against their quantity. The sweep below shows that, with 10 required votes, a 5% threshold gives the 15th most common shelf 10,511 positive books, a 10% threshold gives it 3,105, and a 20% threshold only 376. A low threshold admits shelves that only a handful of readers used, while a high threshold leaves too few positive examples to learn from. We use 10%, which keeps several thousand positive examples for every shelf.

In [17]:
EXCLUDE = {
    'to-read', 'currently-reading', 'read', 'favorites', 'favorite', 'favourites', 'owned', 'own',
    'my-books', 'books-i-own', 'default', 'library', 'wish-list', 'wishlist', 'to-buy',
    'ebook', 'ebooks', 'e-book', 'kindle', 'audiobook', 'audiobooks', 'dnf', 'abandoned',
    'did-not-finish', 're-read', 'romance', 'fiction',
}
MIN_VOTES = 10
MIN_FRAC = 0.10
K = 15

In [18]:
def content_shelves(shelves):
    return {s['name']: int(s['count']) for s in shelves if s['name'] not in EXCLUDE}

df['sh'] = df['popular_shelves'].map(content_shelves)
df['sh_votes'] = df['sh'].map(lambda d: sum(d.values()))
has_text = df['description'].str.strip() != ''

In [19]:
rows = []
for votes in (5, 10, 20):
    keep = df[has_text & (df['sh_votes'] >= votes)]
    for frac in (0.05, 0.10, 0.20, 0.30):
        c = Counter(n for d, v in zip(keep['sh'], keep['sh_votes']) for n, k in d.items() if k / v >= frac)
        pos = [p for _, p in c.most_common(K)]
        rows.append({'min_votes': votes, 'min_frac': frac, 'books': len(keep),
                     'positives_top1': pos[0], f'positives_top{K}': pos[-1]})
pd.DataFrame(rows)

,min_votes,min_frac,books,positives_top1,positives_top15
0,5,0.05,303901,52825,10545
1,5,0.10,303901,19952,3255
2,5,0.20,303901,9459,376
3,5,0.30,303901,3398,38
4,10,0.05,299789,52408,10511
5,10,0.10,299789,19926,3105
6,10,0.20,299789,9310,376
7,10,0.30,299789,3392,31
8,20,0.05,285743,50103,10251
9,20,0.10,285743,19812,2839


In [20]:
elig = df[has_text & (df['sh_votes'] >= MIN_VOTES)].reset_index(drop=True)
# editions of one work share a description, so they must land in the same split
valid_work = ~elig['work_id'].isin(['', '0'])
elig['group'] = np.where(valid_work, 'work:' + elig['work_id'], 'book:' + elig['book_id'])
print(f'{len(df)} raw, {has_text.sum()} with description, {len(elig)} eligible')

335449 raw, 304837 with description, 299789 eligible


In [21]:
dev, te = next(GroupShuffleSplit(test_size=0.2, random_state=0).split(elig, groups=elig['group']))
i_tr, i_va = next(GroupShuffleSplit(test_size=0.2, random_state=1).split(dev, groups=elig['group'].iloc[dev]))
tr, va = dev[i_tr], dev[i_va]
assert not set(elig['group'].iloc[tr]) & set(elig['group'].iloc[te])
print(f'{len(tr)} train, {len(va)} validation, {len(te)} test')

191692 train, 47687 validation, 60410 test


In [22]:
c = Counter(n for d, v in zip(elig['sh'].iloc[tr], elig['sh_votes'].iloc[tr]) for n, k in d.items() if k / v >= MIN_FRAC)
SHELVES = [n for n, _ in c.most_common(K)]
share = np.array([[d.get(s, 0) / v for s in SHELVES] for d, v in zip(elig['sh'], elig['sh_votes'])])
Y = (share >= MIN_FRAC).astype(np.float32)
print(f'no label: {(Y.sum(1) == 0).mean():.1%} of books; negative book-shelf pairs: {1 - Y.mean():.1%}')
pd.DataFrame({'train': Y[tr].mean(0), 'validation': Y[va].mean(0), 'test': Y[te].mean(0)}, index=SHELVES).round(3)

no label: 65.6% of books; negative book-shelf pairs: 97.1%


,train,validation,test
m-m,0.068,0.064,0.064
historical-romance,0.065,0.065,0.062
paranormal,0.045,0.042,0.043
historical,0.039,0.038,0.038
contemporary-romance,0.028,0.026,0.027
paranormal-romance,0.027,0.025,0.025
contemporary,0.027,0.029,0.030
chick-lit,0.026,0.023,0.028
new-adult,0.023,0.021,0.026
young-adult,0.018,0.018,0.022


After filtering, 299,789 books remain: 191,692 for training, 47,687 for validation, and 60,410 for testing. The labels are sparse. 65.6% of the books carry none of the 15 shelves, the most common shelf (`m-m`) applies to only 6.8% of training books, and the least common (`fantasy`) applies to 1.1%. Shelf frequencies are nearly identical across the three splits, so the split does not shift the label distribution.

Some of the 15 shelves are near-synonyms, such as `historical` and `historical-romance`, because readers use both names. We keep them as separate labels because they are separate shelves on Goodreads, but the overlap is one reason to expect some labels to be unreliable.

### 2.3 Measuring label noise

A fixed 10% threshold draws a hard line through what is really a continuous vote share. A book that receives 9% of its votes as `contemporary` is labeled negative, while one at 11% is labeled positive, even though readers treated the two books almost the same way. When many books sit just below the line, the labels for that shelf are unstable. We measure this for each shelf as the number of training books just below the threshold (5% to 10% of votes) divided by the number at or above it. A ratio near zero means readers agree about which books belong on the shelf, and a high ratio means that many books split their readers.

In [23]:
near = ((share[tr] >= 0.05) & (share[tr] < MIN_FRAC)).sum(0) / Y[tr].sum(0)
pd.Series(near, index=SHELVES).sort_values().round(2)

manga                   0.03
m-m                     0.43
historical-romance      0.46
young-adult             0.78
chick-lit               1.03
historical              1.12
bdsm                    1.15
romantic-suspense       1.30
new-adult               1.33
paranormal              1.43
paranormal-romance      1.61
erotica                 2.78
fantasy                 3.30
contemporary-romance    3.61
contemporary            5.51
dtype: float64

The ratios differ by two orders of magnitude. For `manga` the ratio is 0.03, because readers either shelve a book as manga or they do not. For `contemporary` it is 5.51, which means that for every book labeled `contemporary` there are about five more just under the threshold that are labeled negative. `contemporary-romance` (3.61), `fantasy` (3.30), and `erotica` (2.78) follow. We hypothesize that these high-ratio shelves will be the hardest to predict regardless of the model, because their labels depend partly on where an arbitrary line falls.

## 3. Modeling

### 3.1 Formulation

We formulate the task as 15 independent binary classification problems that share a single representation of the text. For shelf $j$ and a book with feature vector $x$, the model computes a score $s_j(x) = w_j^\top x + b_j$ and predicts the shelf when the score exceeds a threshold $t_j$. Treating the shelves independently ignores correlations between them, such as the overlap between `historical` and `historical-romance`, but it keeps the model simple to train and to interpret.

The next cells define the evaluation code and run the two baselines, which require no training. `best_cutoff` chooses, for one shelf, the threshold that maximizes F1 on the validation set by scanning every distinct score. `scores` computes the metrics from section 1 on the test set.

In [24]:
def best_cutoff(y, s):
    """Score cutoff with the highest F1 on these labels."""
    order = np.argsort(-s, kind='stable')
    tp = np.cumsum(y[order])
    f1 = 2 * tp / (np.arange(1, len(y) + 1) + y.sum())
    return s[order][np.argmax(f1)]


def scores(y, s, cut):
    p = s >= cut
    return {'micro_f1': f1_score(y, p, average='micro', zero_division=0),
            'macro_f1': f1_score(y, p, average='macro', zero_division=0),
            'macro_auc': roc_auc_score(y, s, average='macro'),
            'macro_ap': average_precision_score(y, s, average='macro'),
            'labels_per_book': p.sum(1).mean()}


results = {}

In [25]:
prior = Y[tr].mean(0)
results['label prior'] = scores(Y[te], np.tile(prior, (len(te), 1)), 0.5)

In [26]:
def keyword(texts):
    norm = [' ' + re.sub(r'[^\w]+', ' ', t.lower()) + ' ' for t in texts]
    return np.array([[' ' + s.replace('-', ' ') + ' ' in t for s in SHELVES] for t in norm], dtype=float)

results['shelf name in text'] = scores(Y[te], keyword(elig['description'].iloc[te]), 0.5)

### 3.2 Text representation

We represent each description with TF-IDF features. The term frequency counts how often a term appears in a description, and the inverse document frequency down-weights terms that appear in many descriptions. Words that appear in nearly every romance blurb, such as "love" or "life", therefore contribute little, while distinctive words such as "duke" or "vampire" contribute more. We use two kinds of terms. Word unigrams and bigrams (50,000 features, with English stop words removed) capture topical vocabulary and multi-word names such as series titles. Character 3- to 5-grams (60,000 features, given half weight) capture fragments of words, which help with names, inflected forms, and descriptions in other languages. Both vectorizers are fit on the training descriptions only.

This representation assumes that genre is expressed by which words appear, independent of their order. The assumption is reasonable for genres with a characteristic vocabulary, but it cannot capture meaning that depends on word order, or information that a description leaves out.

In [27]:
word = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), min_df=5, stop_words='english',
                       sublinear_tf=True, dtype=np.float32)
char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5, max_features=60_000,
                       sublinear_tf=True, dtype=np.float32)
text = elig['description']
word.fit(text.iloc[tr])
char.fit(text.iloc[tr])


def features(idx):
    return hstack([word.transform(text.iloc[idx]), 0.5 * char.transform(text.iloc[idx])]).tocsr()


X_tr, X_va, X_te = features(tr), features(va), features(te)
FEATURES = np.r_[word.get_feature_names_out(), char.get_feature_names_out()]
print(X_tr.shape, f'{X_tr.nnz / X_tr.shape[0]:.0f} nonzero features per book')

(191692, 110000) 1236 nonzero features per book


### 3.3 Classifier

We train one logistic regression per shelf, implemented as a single linear layer with 15 outputs in PyTorch so that it runs on a GPU. The loss is the binary cross-entropy over all shelves plus an L2 penalty on the weights, which is the objective of scikit-learn's `LogisticRegression(C=1, class_weight='balanced')` divided by the number of training books. Balanced class weights increase the loss on positive examples in proportion to their rarity. Without them, the model could reach a low loss by predicting no shelves at all, since 97% of the targets are negative. We optimize with Adam on mini-batches of 4,096 books for 8 epochs. Each book uses only about 1,200 of the 110,000 features, so we keep the feature matrix in sparse form, which fits in the memory of an 8 GB laptop GPU.

### Choice of model

We chose logistic regression on TF-IDF features because it is a well-established baseline for text classification [3], it is part of the course material, it trains in minutes on 190,000 documents, and its weights can be read directly, which lets us check in section 4 whether the model relies on sensible evidence. In earlier experiments, our group compared eight scikit-learn configurations on validation macro-F1. They varied the features (words only, or words and characters), the regularization strength (`C` of 0.25, 1, or 4), class weighting, and the classifier (logistic regression or a linear SVM). Logistic regression with word and character features and `C=1` performed best, and this notebook uses that configuration. Models that read word order, such as fine-tuned transformers, might capture more of a description's meaning, but they are far more expensive to train on this many documents, and we did not evaluate them.

In [28]:
# Same objective as sklearn's LogisticRegression(C=1, class_weight='balanced'), divided by n:
# mean weighted log-loss + ||W||^2 / (2 C n). Mini-batches keep memory small on an 8 GB card.
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'


def to_torch(x):
    x = x.tocoo()
    i = torch.from_numpy(np.vstack([x.row, x.col]).astype(np.int64))
    return torch.sparse_coo_tensor(i, torch.from_numpy(x.data), x.shape).to(DEV)


def fit_lr(x, y, C=1.0, epochs=8, batch=4096, lr=0.01, seed=0):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    n = len(y)
    yt = torch.from_numpy(y).to(DEV)
    pos = yt.sum(0)
    w_pos, w_neg = n / (2 * pos), n / (2 * (n - pos))  # class_weight='balanced'
    W = torch.zeros(x.shape[1], y.shape[1], device=DEV, requires_grad=True)
    b = torch.zeros(y.shape[1], device=DEV, requires_grad=True)
    opt = torch.optim.Adam([W, b], lr=lr)
    for epoch in range(epochs):
        total = 0.0
        for idx in np.array_split(rng.permutation(n), max(n // batch, 1)):
            yb = yt[idx]
            logits = torch.sparse.mm(to_torch(x[idx]), W) + b
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                logits, yb, weight=yb * w_pos + (1 - yb) * w_neg)
            loss = loss + (W ** 2).sum() / (2 * C * n)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        print(f'epoch {epoch + 1}: loss {total / n:.4f}')
    return W.detach(), b.detach()


@torch.no_grad()
def decision(x, W, b, batch=8192):
    out = [(torch.sparse.mm(to_torch(x[i:i + batch]), W) + b).cpu().numpy() for i in range(0, x.shape[0], batch)]
    return np.vstack(out)


W, b = fit_lr(X_tr, Y[tr])
S_va, S_te = decision(X_va, W, b), decision(X_te, W, b)

C:\Users\jdoan\AppData\Local\Temp\ipykernel_23452\1004077017.py:9: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  return torch.sparse_coo_tensor(i, torch.from_numpy(x.data), x.shape).to(DEV)


epoch 1: loss 0.5337
epoch 2: loss 0.4107
epoch 3: loss 0.3831
epoch 4: loss 0.3698
epoch 5: loss 0.3620
epoch 6: loss 0.3571
epoch 7: loss 0.3538
epoch 8: loss 0.3515


The training loss is still decreasing at the eighth epoch, so this model has not fully converged. In our earlier experiments, scikit-learn's solver fit the same objective to convergence and reached a test macro-F1 of 0.484, against 0.454 here. The results below are therefore a slight underestimate of what this model can achieve.

## 4. Evaluation

### 4.1 Overall results

We now choose each shelf's threshold on the validation set and evaluate every model once on the test set.

In [29]:
cut = np.array([best_cutoff(Y[va][:, j], S_va[:, j]) for j in range(K)])
results['model, cutoff 0'] = scores(Y[te], S_te, 0.0)
results['model, cutoff per shelf'] = scores(Y[te], S_te, cut)
pd.DataFrame(results).T.round(3)

,micro_f1,macro_f1,macro_auc,macro_ap,labels_per_book
label prior,0.000,0.000,0.500,0.029,0.000
shelf name in text,0.108,0.106,0.536,0.051,0.148
"model, cutoff 0",0.350,0.341,0.933,0.397,1.588
"model, cutoff per shelf",0.480,0.454,0.933,0.397,0.611


The results answer the first part of our question: the description carries substantial information about a book's shelves, far more than an obvious keyword rule can extract. The label prior scores zero, as expected. Searching the description for the shelf's name reaches a macro-F1 of only 0.106, because reader-invented shelf names rarely appear in publisher text; a book that readers shelve as `contemporary` almost never calls itself contemporary. The learned model reaches a macro-F1 of 0.454 and a micro-F1 of 0.480, more than four times the keyword baseline.

Threshold tuning matters as much as the learned weights. With the default threshold of 0.5, the balanced class weights make the model generous: it predicts 1.59 shelves per book, while books carry about 0.44 on average, and its macro-F1 is 0.341. Thresholds tuned on validation reduce this to 0.61 shelves per book and raise macro-F1 to 0.454. AUC does not depend on the threshold, which is why it is identical in both rows.

The gap between AUC and AP shows what kind of errors remain. An AUC of 0.933 means that, for a given shelf, a randomly chosen book that has the shelf receives a higher score than a randomly chosen book that does not about 93% of the time, averaged over shelves. In other words, the model ranks books well. AP, at 0.397, reflects how many of the highest-ranked books are correct, and it is much lower because positives are rare. Even a small error rate on the large number of negative books is enough to fill the top of the ranking with false positives.

In [30]:
P = S_te >= cut
per_shelf = pd.DataFrame({
    'test_positives': Y[te].sum(0).astype(int),
    'f1': f1_score(Y[te], P, average=None, zero_division=0),
    'precision': precision_score(Y[te], P, average=None, zero_division=0),
    'recall': recall_score(Y[te], P, average=None, zero_division=0),
    'auc': [roc_auc_score(Y[te][:, j], S_te[:, j]) for j in range(K)],
    'near_cutoff_ratio': near,
}, index=SHELVES)
per_shelf.sort_values('f1').round(3)

,test_positives,f1,precision,recall,auc,near_cutoff_ratio
contemporary,1833,0.244,0.190,0.343,0.834,5.507
contemporary-romance,1635,0.279,0.241,0.330,0.859,3.608
erotica,983,0.308,0.232,0.459,0.915,2.781
fantasy,625,0.328,0.357,0.302,0.933,3.302
new-adult,1549,0.339,0.382,0.304,0.921,1.330
young-adult,1304,0.393,0.336,0.475,0.945,0.781
romantic-suspense,876,0.401,0.309,0.572,0.954,1.295
chick-lit,1720,0.422,0.397,0.451,0.928,1.035
paranormal,2596,0.459,0.358,0.639,0.931,1.425
paranormal-romance,1494,0.460,0.374,0.598,0.946,1.611


### 4.2 Results by shelf

The average hides large differences between shelves. `manga` reaches an F1 of 0.812: 86% of the books the model shelves as manga are manga, and it finds 77% of all manga in the test set. `historical-romance` (0.685) and `m-m` (0.627) also perform well. At the other end, `contemporary` reaches an F1 of 0.244, with only 19% of its predictions correct, and `contemporary-romance` reaches 0.279. These two shelves also have the lowest AUC (0.834 and 0.859). A low AUC means the model cannot separate their books from the rest even in its ranking, so no choice of threshold would fix them.

To the question of whether the model can predict a book's shelf from its description, the per-shelf results give a split answer. For genres with a distinctive vocabulary it can, with useful precision and recall. For genres such as `contemporary` it largely cannot. The next cell tests why.

In [31]:
rho_noise = spearmanr(per_shelf['near_cutoff_ratio'], per_shelf['f1'])[0]
rho_count = spearmanr(per_shelf['test_positives'], per_shelf['f1'])[0]
print(f'rank correlation with F1: near-cutoff ratio {rho_noise:+.2f}, test positives {rho_count:+.2f}')

rank correlation with F1: near-cutoff ratio -0.82, test positives +0.23


### 4.3 Why some shelves fail

We test the hypothesis from section 2.3 by comparing each shelf's label-noise ratio with its test F1. The Spearman rank correlation across the 15 shelves is -0.82: the shelves on which readers disagree most are the shelves the model predicts worst. The number of positive test books, in contrast, is only weakly related to F1 (+0.23), so the weak shelves are not weak simply because they are small. For example, `contemporary` has 1,833 positive test books and `historical` has 2,307, a similar number, yet `historical` reaches an F1 of 0.525 and `contemporary` only 0.244.

This suggests that the ceiling on these shelves comes mainly from the labels. If readers themselves do not agree on whether a book is `contemporary`, a classifier trained to reproduce their votes inherits that disagreement. With only 15 shelves, the correlation is suggestive and not conclusive, and it does not rule out other explanations, such as the lack of a distinctive vocabulary that we turn to next.

In [32]:
Wn = W.cpu().numpy()
top = pd.DataFrame([FEATURES[np.argsort(-Wn[:, j])[:8]] for j in range(K)], index=SHELVES)
print(top.to_string(header=False))

m-m                            gay     his       his      man      lover         men        his            he 
historical-romance            lady     earl     lord     duke     london  historical    viscount         marry
paranormal              paranormal     mate  vampire  psychic    shifter       demon        vamp           vam
historical                    lady   london     earl     lord       duke         war  historical      viscount
contemporary-romance  virgin river     sexy    idaho    river  fool gold     gansett        town         susan
paranormal-romance         vampire     vamp      vam     vamp        vam       vampi       ampir         demon
contemporary                  lucy      dad      guy    idaho     walker          .        maybe  lucky harbor
chick-lit                       -     holly    novel  bridget  boyfriend      mother   hilarious           job
new-adult                new adult  college     girl    adult     campus  university      hardin           guy
y

### 4.4 What the model learned

The table above lists the eight terms with the largest positive weight for each shelf. For the well-predicted shelves, these are the terms a human reader would expect. `historical-romance` relies on lady, earl, lord, and duke; `bdsm` on dom, master, and submissive; `manga` on the word manga and fragments of Japanese names; and `m-m` on gay and male pronouns. The model uses genre evidence on these shelves, which supports the validity of its predictions there.

For `contemporary` and `contemporary-romance`, the strongest terms are the names of particular series and places, such as Virgin River, Idaho, and Lucky Harbor. A contemporary romance is marked mainly by the absence of other markers, a present-day setting with no historical or supernatural elements, so the genre has little vocabulary of its own, and the model falls back on memorizing popular series. That strategy cannot generalize to a standalone novel. A few character-level features, such as single punctuation marks, also appear among the top terms, which suggests the model picks up formatting habits of particular publishers.

### 4.5 Errors and limitations

In an earlier run of the same kind of model, our group read ten of the most confident errors alongside their descriptions and vote counts. Most of them were not failures to understand the text. In some cases, readers had split their votes across spellings (`manga`, `mangas`, `manga-manhwa`), so the exact shelf stayed under 10% even though the description fits it. In others, a sibling shelf such as `contemporary-romance` was positive while `contemporary` fell just below the threshold, or the label rested on two to five votes for a book whose description was about something else. Three of the ten were cases where the description never mentioned the relevant fact, such as a manga volume whose stored description is a prose plot summary. Ten examples chosen for high confidence show which kinds of errors occur, but not how often each kind occurs.

The setup has further limitations. The labels are a single snapshot of reader votes that nobody verified, so any measured performance is bounded by their quality. The books we evaluate already have at least 10 genre votes and are more established than a typical new release. The model ignores correlations between shelves, and we keep near-duplicate shelves separate. In the earlier run, merging the three pairs of near-duplicate shelves into 12 labels raised the scores, but that answers an easier question and does not reflect a better model.

### 4.6 Answer to the research question

Based on its description alone, a TF-IDF model can predict a romance book's shelves well when the shelf corresponds to a distinctive vocabulary, as with manga, historical romance, or m/m romance, where F1 ranges from 0.63 to 0.81. It predicts poorly when the shelf is one that readers apply inconsistently, as with contemporary romance, where F1 stays below 0.3. Across all 15 shelves, the model scores more than four times higher than a keyword rule, and the strongest predictor of where it fails is disagreement among readers about the label, not the amount of training data. For a cold-start system, this means that descriptions are a reliable source of genre tags for some shelves and an unreliable one for others.

## 5. Related work

The dataset was introduced by Wan and McAuley [1], who used Goodreads interactions to study recommendation when a user's actions form a monotonic chain: shelving a book, reading it, rating it, and recommending it. Their models learn from users' past interactions with each item, which a newly released book does not yet have. Our work addresses the complementary, item-side problem of characterizing a new book from its own content. Wan et al. [2] used reviews from the same crawl for fine-grained spoiler detection, a classification task on review sentences. Our romance file contains descriptions and no reviews, so we cannot compare with that task directly.

On the modeling side, Wang and Manning [3] showed that linear classifiers over word unigram and bigram features, such as Naive Bayes and linear SVMs, are strong baselines for sentiment and topic classification, and that bigrams help. Our results are consistent with theirs. A linear model over TF-IDF features performs far above trivial baselines, and in our group's earlier experiments, adding character n-grams to word features raised macro-F1 by 0.022. Wang and Manning also note that bag-of-words models cannot capture meaning that arises from how words combine. Our error analysis points to a different limit: many errors come from the labels themselves, or from information that is absent from the description and that no text model could recover.

We are not aware of published results for shelf prediction on this dataset, so we cannot compare our F1 scores with prior work directly; our comparison points are the baselines in section 4.

### References

[1] Mengting Wan and Julian McAuley. 2018. Item Recommendation on Monotonic Behavior Chains. In *Proceedings of the 12th ACM Conference on Recommender Systems (RecSys '18)*. ACM, 86-94. https://doi.org/10.1145/3240323.3240369

[2] Mengting Wan, Rishabh Misra, Ndapa Nakashole, and Julian McAuley. 2019. Fine-Grained Spoiler Detection from Large-Scale Review Corpora. In *Proceedings of the 57th Annual Meeting of the Association for Computational Linguistics*, 2605-2610. https://aclanthology.org/P19-1248/

[3] Sida Wang and Christopher D. Manning. 2012. Baselines and Bigrams: Simple, Good Sentiment and Topic Classification. In *Proceedings of the 50th Annual Meeting of the Association for Computational Linguistics (Volume 2: Short Papers)*, 90-94. https://aclanthology.org/P12-2018/